In [ ]:
# Group 07 - Task 1: Exploratory Data Analysis (EDA)
# Dataset: UCI EMG Physical Action Dataset (ID 213)

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from scipy.stats import skew, kurtosis
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from ucimlrepo import fetch_ucirepo

warnings.filterwarnings('ignore')

# --- LOAD DATASET ---
print('Loading UCI EMG Physical Action Dataset...')
emg_data = fetch_ucirepo(id=213)
X = emg_data.data.features
y = emg_data.data.targets

channel_names = ['R_Bicep', 'R_Tricep', 'L_Bicep', 'L_Tricep', 'R_Thigh', 'R_Hamstring', 'L_Thigh', 'L_Hamstring']
X.columns = channel_names

# --- A. SUMMARY TABLE ---
summary_df = pd.DataFrame({
    'Metric': ['Dataset Name', 'Total Samples', 'Channels (Nodes)', 'Classes', 'Missing Values %', 'Duplicates'],
    'Value': ['UCI EMG Physical Action', f'{X.shape[0]:,}', X.shape[1], 20, f'{X.isnull().sum().sum()/X.size*100:.2f}%', X.duplicated().sum()]
})
print('=== A. SUMMARY TABLE ===')
print(summary_df.to_string(index=False))

# --- B. STATISTICAL PROFILE ---
stats_df = pd.DataFrame({
    'Mean': X.mean(), 'Std': X.std(), 'Min': X.min(),
    'Max': X.max(), 'Skewness': X.skew(), 'Kurtosis': X.kurtosis()
})
print('\n=== B. STATISTICAL PROFILE ===')
print(stats_df.round(4))

# --- C. DATA QUALITY & OUTLIERS ---
plt.figure(figsize=(10, 4))
sns.boxplot(data=X, palette='Set3')
plt.title('C. sEMG Signal Amplitude Distributions & Outlier Artifacts')
plt.ylabel('Microvolts (uV)')
plt.tight_layout()
plt.show()

# --- D. CLASS BALANCE ---
plt.figure(figsize=(10, 4))
y_counts = y.iloc[:, 0].value_counts()
sns.barplot(x=y_counts.index, y=y_counts.values, palette='viridis')
plt.title('D. Action Class Distribution (20 Physical Actions)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- E. FEATURE DISTRIBUTIONS (WINDOWED RMS) ---
window_size = 200
X_rms = np.sqrt(X.groupby(np.arange(len(X)) // window_size).apply(lambda g: (g**2).mean()))
plt.figure(figsize=(10, 4))
sns.violinplot(data=X_rms, palette='coolwarm')
plt.title('E. Node Feature Distribution: Windowed RMS Energy per Channel')
plt.tight_layout()
plt.show()

# --- F. CORRELATION HEATMAP (GNN ADJACENCY BASE) ---
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, cmap='mako', fmt='.2f', vmin=-1, vmax=1)
plt.title('F. Inter-Channel Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

# --- G. PCA & t-SNE MANIFOLD ---
X_sub = X.iloc[:3000]
y_sub = y.iloc[:3000, 0].astype('category').cat.codes
pca_res = PCA(n_components=2).fit_transform(X_sub)
tsne_res = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(pca_res[:, 0], pca_res[:, 1], c=y_sub, cmap='tab20', alpha=0.5, s=8)
axes[0].set_title('G1. 2D PCA Projection')
axes[1].scatter(tsne_res[:, 0], tsne_res[:, 1], c=y_sub, cmap='tab20', alpha=0.5, s=8)
axes[1].set_title('G2. 2D t-SNE Manifold')
plt.tight_layout()
plt.show()

# --- H. INTERACTIVE TIME SERIES ---
fig_plotly = go.Figure()
for col in X.columns[:4]:
    fig_plotly.add_trace(go.Scatter(y=X[col].iloc[:1000], mode='lines', name=col))
fig_plotly.update_layout(title='H. Multi-Channel sEMG Time Series (First 1000 Samples)', xaxis_title='Time Step', yaxis_title='uV')
fig_plotly.show()
